# Escalações e Copas

## Task 4: conferir as escalações e o resumo oficial de cada Copa e levar o código para o projeto

## Os três arquivos juntos

- `WorldCupMatches.csv`: uma linha por partida
- `WorldCupPlayers.csv`: uma linha por jogador em cada partida
- `WorldCups.csv`: uma linha por Copa, com sede, campeão e o resumo de partidas e gols

In [8]:
import sys
sys.path.insert(0, "../../src")

from pyspark.sql import SparkSession, functions as F

spark = (
    SparkSession.builder
    .appName("arena-joins")
    .master("local[*]")
    .config("spark.sql.warehouse.dir", "../../data/warehouse")
    .getOrCreate()
)

RAW = "../../data/raw"

spark

A célula abaixo já vem pronta: o mesmo código dos vídeos 2.1 e 2.2.

In [9]:
# vídeo 2.1: leitura e limpeza
from wc_pipeline.schemas import MATCHES

partidas_brutas = (
    spark.read
    .option("header", "true")
    .schema(MATCHES)
    .csv(f"{RAW}/WorldCupMatches.csv")
)

partidas_limpas = (
    partidas_brutas
    .filter(F.col("MatchID").isNotNull())
    .dropDuplicates(["MatchID"])
)

# vídeo 2.2: padronização
partidas = partidas_limpas.select(
    F.col("Year").alias("ano"),
    F.col("MatchID").alias("match_id"),
    F.col("Stage").alias("fase"),
    F.col("Home Team Name").alias("mandante"),
    F.col("Home Team Goals").alias("gols_mandante"),
    F.col("Away Team Goals").alias("gols_visitante"),
    F.col("Away Team Name").alias("visitante"),
    F.col("Attendance").alias("publico"),
    F.col("Datetime").alias("data_partida"),
)

partidas = partidas.withColumn(
    "publico",
    F.regexp_replace("publico", ",", "").cast("int"),
)

FORMATO_DATA = "dd MMM yyyy - HH:mm"

partidas = partidas.withColumn(
    "data_partida",
    F.to_timestamp("data_partida", FORMATO_DATA),
)

partidas = (
    partidas
    .withColumn("total_gols", F.col("gols_mandante") + F.col("gols_visitante"))
    .withColumn(
        "resultado",
        F.when(F.col("gols_mandante") > F.col("gols_visitante"), "mandante")
         .when(F.col("gols_mandante") < F.col("gols_visitante"), "visitante")
         .otherwise("empate"),
    )
)

partidas.printSchema()

root
 |-- ano: integer (nullable = true)
 |-- match_id: integer (nullable = true)
 |-- fase: string (nullable = true)
 |-- mandante: string (nullable = true)
 |-- gols_mandante: integer (nullable = true)
 |-- gols_visitante: integer (nullable = true)
 |-- visitante: string (nullable = true)
 |-- publico: integer (nullable = true)
 |-- data_partida: timestamp (nullable = true)
 |-- total_gols: integer (nullable = true)
 |-- resultado: string (nullable = false)



In [10]:
from wc_pipeline.schemas import PLAYERS, WORLD_CUPS

escalacoes = (
    spark.read.option("header", "true").schema(PLAYERS).csv(f"{RAW}/WorldCupPlayers.csv")
    .select(
        F.col("MatchID").alias("match_id"),
        F.col("Team Initials").alias("sigla"),
        F.col("Player Name").alias("jogador"),
        F.col("Position").alias("posicao"),
    )
)

copas = (
    spark.read.option("header", "true").schema(WORLD_CUPS).csv(f"{RAW}/WorldCups.csv")
    .select(
        F.col("Year").alias("ano"),
        F.col("Country").alias("sede"),
        F.col("Winner").alias("campeao"),
        F.col("MatchesPlayed").alias("partidas_oficiais"),
        F.col("GoalsScored").alias("gols_oficiais"),
    )
)

print("partidas:", partidas.count(), "| escalações:", escalacoes.count(), "| copas:", copas.count())

partidas: 964 | escalações: 29183 | copas: 22


## Os tipos de join

| Tipo | O que devolve | Pergunta |
|---|---|---|
| `inner` | só as linhas com par nos dois lados | "partidas **com** escalações" |
| `left` | tudo de A; colunas de B `null` sem par | "todas as Copas, com o resumo **se houver**" |
| `left_anti` | linhas de A **sem** par em B | "quais partidas **não têm** escalação?" (`NOT EXISTS`) |
| `left_semi` | linhas de A **com** par em B | "quais partidas **têm** escalação?" (`EXISTS`) |

## left_anti: as escalações conversam com as partidas?

Partidas sem escalação, e escalações sem partida:

In [11]:
partidas_sem_escalacao = partidas.join(escalacoes, on="match_id", how="left_anti")
escalacoes_sem_partida = escalacoes.join(partidas, on="match_id", how="left_anti")

print("partidas sem escalação:", partidas_sem_escalacao.count())
print("escalações sem partida:", escalacoes_sem_partida.count())

partidas sem escalação: 0
escalações sem partida: 0


A demanda não pede estatísticas com as escalações: o arquivo foi conferido e não entra nas próximas tasks.

## left join: os números batem com o resumo oficial?

`copas` é o resumo oficial de cada Copa. Depois da limpeza, as partidas e os gols de cada Copa precisam bater com ele.

In [12]:
gols_por_copa = partidas.groupBy("ano").agg(
    F.count("*").alias("partidas"),
    F.sum("total_gols").alias("gols"),
)

conferencia = gols_por_copa.join(copas, on="ano", how="left")
conferencia.select("ano", "sede", "partidas", "partidas_oficiais", "gols", "gols_oficiais").orderBy("ano").show()

+----+-------------------+--------+-----------------+----+-------------+
| ano|               sede|partidas|partidas_oficiais|gols|gols_oficiais|
+----+-------------------+--------+-----------------+----+-------------+
|1930|            Uruguay|      18|               18|  70|           70|
|1934|              Italy|      17|               17|  70|           70|
|1938|             France|      18|               18|  84|           84|
|1950|             Brazil|      22|               22|  88|           88|
|1954|        Switzerland|      26|               26| 140|          140|
|1958|             Sweden|      35|               35| 126|          126|
|1962|              Chile|      32|               32|  89|           89|
|1966|            England|      32|               32|  89|           89|
|1970|             Mexico|      32|               32|  95|           95|
|1974|       West Germany|      38|               38|  97|           97|
|1978|          Argentina|      38|               3

In [13]:
divergentes = conferencia.filter(
    (F.col("partidas") != F.col("partidas_oficiais")) | (F.col("gols") != F.col("gols_oficiais"))
)
print("Copas que não batem com o resumo oficial:", divergentes.count())

Copas que não batem com o resumo oficial: 0


Cada Copa casa com exatamente uma linha do resumo: continua uma linha por Copa.

## Levando o código para o projeto

`%%writefile` grava o conteúdo da célula num arquivo. Em `src/wc_pipeline/transformacoes.py`, o código da aula, sem
nenhuma linha nova:

| Função | De onde vem o código |
|---|---|
| `limpar_partidas` | vídeos 2.1 e 2.2 |
| `partidas_por_selecao` | vídeo 2.3 (`unionByName` e V, E ou D) |
| `resumo_por_selecao` | vídeo 2.3 (a agregação da `tabela_selecoes`) |

In [14]:
%%writefile ../../src/wc_pipeline/transformacoes.py
from pyspark.sql import DataFrame, functions as F

FORMATO_DATA = "dd MMM yyyy - HH:mm"

def limpar_partidas(partidas_brutas: DataFrame) -> DataFrame:
    # video 2.1: limpeza
    partidas_limpas = (
        partidas_brutas
        .filter(F.col("MatchID").isNotNull())
        .dropDuplicates(["MatchID"])
    )

    # video 2.2: padronizacao
    partidas = partidas_limpas.select(
        F.col("Year").alias("ano"),
        F.col("MatchID").alias("match_id"),
        F.col("Stage").alias("fase"),
        F.col("Home Team Name").alias("mandante"),
        F.col("Home Team Goals").alias("gols_mandante"),
        F.col("Away Team Goals").alias("gols_visitante"),
        F.col("Away Team Name").alias("visitante"),
        F.col("Attendance").alias("publico"),
        F.col("Datetime").alias("data_partida"),
    )

    partidas = partidas.withColumn(
        "publico",
        F.regexp_replace("publico", ",", "").cast("int"),
    )

    partidas = partidas.withColumn(
        "data_partida",
        F.to_timestamp("data_partida", FORMATO_DATA),
    )

    partidas = (
        partidas
        .withColumn("total_gols", F.col("gols_mandante") + F.col("gols_visitante"))
        .withColumn(
            "resultado",
            F.when(F.col("gols_mandante") > F.col("gols_visitante"), "mandante")
             .when(F.col("gols_mandante") < F.col("gols_visitante"), "visitante")
             .otherwise("empate"),
        )
    )
    return partidas


def partidas_por_selecao(partidas: DataFrame) -> DataFrame:
    como_mandante = partidas.select(
        "ano",
        F.col("mandante").alias("selecao"),
        F.col("gols_mandante").alias("gols_pro"),
        F.col("gols_visitante").alias("gols_contra"),
    )
    como_visitante = partidas.select(
        "ano",
        F.col("visitante").alias("selecao"),
        F.col("gols_visitante").alias("gols_pro"),
        F.col("gols_mandante").alias("gols_contra"),
    )
    por_selecao = como_mandante.unionByName(como_visitante)

    por_selecao = por_selecao.withColumn(
        "resultado",
        F.when(F.col("gols_pro") > F.col("gols_contra"), "V")
         .when(F.col("gols_pro") < F.col("gols_contra"), "D")
         .otherwise("E"),
    )
    return por_selecao


def resumo_por_selecao(por_selecao: DataFrame) -> DataFrame:
    tabela_selecoes = (
        por_selecao.groupBy("ano", "selecao")
        .agg(
            F.count("*").alias("jogos"),
            F.sum(F.when(F.col("resultado") == "V", 1).otherwise(0)).alias("vitorias"),
            F.sum(F.when(F.col("resultado") == "E", 1).otherwise(0)).alias("empates"),
            F.sum(F.when(F.col("resultado") == "D", 1).otherwise(0)).alias("derrotas"),
            F.sum("gols_pro").alias("gols_pro"),
            F.sum("gols_contra").alias("gols_contra"),
        )
        .withColumn("saldo", F.col("gols_pro") - F.col("gols_contra"))
    )
    return tabela_selecoes

Writing ../../src/wc_pipeline/transformacoes.py


In [16]:
from wc_pipeline.transformacoes import limpar_partidas, partidas_por_selecao, resumo_por_selecao

limpas = limpar_partidas(partidas_brutas)
resumo = resumo_por_selecao(partidas_por_selecao(limpas))

print("partidas limpas:", limpas.count())
resumo.orderBy(F.desc("saldo")).show(5)

partidas limpas: 964
+----+-------+-----+--------+-------+--------+--------+-----------+-----+
| ano|selecao|jogos|vitorias|empates|derrotas|gols_pro|gols_contra|saldo|
+----+-------+-----+--------+-------+--------+--------+-----------+-----+
|1954|Hungary|    5|       4|      0|       1|      27|         10|   17|
|1950| Brazil|    6|       4|      1|       1|      22|          6|   16|
|2014|Germany|    7|       6|      1|       0|      18|          4|   14|
|2002| Brazil|    7|       7|      0|       0|      18|          4|   14|
|1998| France|    7|       6|      1|       0|      15|          2|   13|
+----+-------+-----+--------+-------+--------+--------+-----------+-----+
only showing top 5 rows


## Demanda da aula 2 atendida

| Etapa | Entrega |
|---|---|
| Task 1 | partidas auditadas e limpas |
| Task 2 | nomes em português e tipos corrigidos |
| Task 3 | tabelas por Copa e por seleção, conferidas pelo total de gols |
| Task 4 | escalações conferidas, partidas e gols batendo com o resumo oficial, e a limpeza em `src/wc_pipeline/transformacoes.py` |

## Para consultar depois
- **Chaves com nomes diferentes:** quando a coluna não tem o mesmo nome nos dois lados, use uma expressão, por
  exemplo `a.join(b, a.ano == b.Year, "left")`. Com uma expressão, o resultado fica com as duas colunas da chave;
  com `on="nome"`, com uma só.
- **`left_semi`:** devolve as linhas da esquerda que **têm** par na direita, só com as colunas da esquerda. É o
  `WHERE EXISTS` do SQL, o contrário do `left_anti`.

## Encerrando

Ao terminar o notebook, encerre a sessão Spark. Isso libera a memória e a porta da Spark UI para o próximo notebook.

In [17]:
spark.stop()